# **Reddit Scraper Notebook for HealthPH+**


# **Dependencies**

In [1]:
import requests
import time
import pandas as pd
from datetime import datetime
from urllib.parse import urlparse, parse_qs
import os
import re


In [3]:
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus
import hashlib
import json

try:
    from playwright.async_api import (
        async_playwright,
        TimeoutError as PlaywrightTimeoutError,
    )
except ImportError as exc:
    raise ImportError(
        "playwright is not installed. Run: pip install playwright && playwright install chromium"
    ) from exc

In [4]:
print("✅ Libraries loaded successfully")

✅ Libraries loaded successfully


## **Reddit**

### Configuration

In [7]:
# ── USER SETTINGS ──────────────────────────────────────────────────────────────
# Paste any Reddit search URL below.
# The query, sort order, and time filter are parsed automatically from the URL.
# Example filters you can append to the URL:
#   &sort=relevance | new | top | comments
#   &t=all | year | month | week | day | hour


KEYWORD = 'masama pakiramdam'
SEARCH_URL = f"https://www.reddit.com/search/?q={KEYWORD}&sort=new&t=all"

LIMIT     = 50   # Posts per page (max 100)
MAX_PAGES = 10   # Number of pages to paginate through

CWD = Path.cwd()
ROOT_DIR = next((p for p in [CWD, *CWD.parents] if (p / '.git').exists()), CWD)
OUTPUT_FILE = str(ROOT_DIR / "data" / "raw" / "reddit" / "reddit_results.csv")   # Master data output file
# ───────────────────────────────────────────────────────────────────────────────

print(f"Search URL : {SEARCH_URL}")
print(f"Limit      : {LIMIT} posts/page")
print(f"Max pages  : {MAX_PAGES}")
print(f"Output file: {OUTPUT_FILE}")

Search URL : https://www.reddit.com/search/?q=masama pakiramdam&sort=new&t=all
Limit      : 50 posts/page
Max pages  : 10
Output file: /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/raw/reddit/reddit_results.csv


In [8]:
## 3. Helper Functions
def extract_reddit_search_params(url):
    """
    Extract search query, sort order, and time filter from a Reddit search URL.
    
    Example URL:
        https://www.reddit.com/search/?q=sakit&sort=top&t=month
    
    Returns:
        tuple: (query, sort, time_filter)
    """
    parsed = urlparse(url)
    params = parse_qs(parsed.query)

    query = params.get("q", [None])[0]
    sort  = params.get("sort", ["relevance"])[0]
    t     = params.get("t",    ["all"])[0]

    if not query:
        raise ValueError(f"Could not extract a search query from URL: {url}")

    return query, sort, t


def get_last_reddit_fullname_from_csv(filepath):
    """
    Read the last saved Reddit post URL from CSV and return its fullname (t3_<id>).

    Returns None if the file is missing, empty, or the URL is not parseable.
    """
    if not os.path.isfile(filepath):
        return None

    try:
        df = pd.read_csv(filepath, usecols=["url"])
        if df.empty:
            return None

        last_url = df["url"].dropna().iloc[-1]
        if not isinstance(last_url, str) or not last_url:
            return None

        match = re.search(r"/comments/([a-z0-9]+)/", last_url)
        if not match:
            return None

        return f"t3_{match.group(1)}"
    except Exception as e:
        print(f"⚠️ Could not read last id from {filepath}: {e}")
        return None


def scrape_reddit_search(url, limit=25, max_pages=3):
    """
    Scrape Reddit search results from a Reddit search URL.

    Args:
        url       : A Reddit search URL.
        limit     : Posts per page (max 100).
        max_pages : Maximum number of pages to paginate through.

    Returns:
        pd.DataFrame with columns: title, selftext, created, ups, subreddit, url
    """
    query, sort, t = extract_reddit_search_params(url)
    print(f"Query: '{query}'  |  Sort: {sort}  |  Time filter: {t}\n")

    headers  = {"User-Agent": "RedditScraper/1.0 (personal research script)"}
    base_url = "https://www.reddit.com/search.json"
    results  = []
    after    = None  # pagination cursor (within this run only)

    for page in range(max_pages):
        params = {
            "q":     query,
            "limit": limit,
            "sort":  sort,
            "t":     t,
            "type":  "link",   # posts only
        }
        if after:
            params["after"] = after

        response = requests.get(base_url, params=params, headers=headers)

        if response.status_code != 200:
            print(f"❌ Error: HTTP {response.status_code}")
            break

        data  = response.json()
        posts = data.get("data", {}).get("children", [])

        if not posts:
            print("ℹ️  No more posts found.")
            break

        for post in posts:
            pd_  = post.get("data", {})
            results.append({
                "title":     pd_.get("title", ""),
                "selftext":  pd_.get("selftext", ""),
                "created":   datetime.fromtimestamp(
                                 pd_.get("created_utc", 0)
                             ).strftime("%Y-%m-%d %H:%M:%S UTC"),
                "ups":       pd_.get("ups", 0),
                "subreddit": pd_.get("subreddit", ""),
                "url":       f"https://reddit.com{pd_.get('permalink', '')}",
            })

        after = data.get("data", {}).get("after")
        print(f"✔ Page {page + 1}: fetched {len(posts)} posts  (running total: {len(results)})")

        if not after:
            print("ℹ️  Reached last page.")
            break

        time.sleep(1)   # polite delay to avoid rate limiting

    return pd.DataFrame(results)


def save_results(df, filepath=None):
    """
    Save Reddit posts to CSV without adding duplicates.
    Dedupe key: post URL.
    """
    if filepath is None:
        filepath = OUTPUT_FILE

    os.makedirs(os.path.dirname(filepath), exist_ok=True)

    if df.empty:
        print("ℹ️ No rows to save.")
        return

    if "url" not in df.columns:
        raise ValueError("save_results requires a 'url' column for deduplication.")

    file_exists = os.path.isfile(filepath)
    incoming_count = len(df)

    to_save = df.copy()
    to_save["url"] = to_save["url"].astype(str).str.strip()
    to_save = to_save[to_save["url"] != ""]

    # Remove duplicates in this scrape batch
    to_save = to_save.drop_duplicates(subset=["url"], keep="first")

    # Remove rows already present in the output file
    if file_exists and not to_save.empty:
        try:
            existing_urls = set(
                pd.read_csv(filepath, usecols=["url"])["url"]
                .dropna()
                .astype(str)
                .str.strip()
            )
            to_save = to_save[~to_save["url"].isin(existing_urls)]
        except ValueError:
            print("⚠️ Existing file has no 'url' column. Skipping cross-run dedupe.")

    new_rows = len(to_save)
    if new_rows == 0:
        skipped = incoming_count
        print(f"ℹ️ No new posts to append. (Skipped {skipped} duplicates)")
    else:
        to_save.to_csv(
            filepath,
            mode="a",
            index=False,
            encoding="utf-8-sig",
            header=not file_exists,
        )
        skipped = incoming_count - new_rows
        action = "Appended to" if file_exists else "Created"
        print(f"💾 {action} {filepath}  (+{new_rows} posts, skipped {skipped} duplicates)")

    total = pd.read_csv(filepath).shape[0] if os.path.isfile(filepath) else 0
    print(f"📊 Total rows in file: {total}")


print("✅ Functions defined")

✅ Functions defined


### Main Function

In [9]:
df = scrape_reddit_search(url=SEARCH_URL, limit=LIMIT, max_pages=MAX_PAGES)
print(f"Total posts collected: {len(df)}")


Query: 'masama pakiramdam'  |  Sort: new  |  Time filter: all

✔ Page 1: fetched 50 posts  (running total: 50)
✔ Page 2: fetched 50 posts  (running total: 100)
✔ Page 3: fetched 50 posts  (running total: 150)
✔ Page 4: fetched 50 posts  (running total: 200)
✔ Page 5: fetched 29 posts  (running total: 229)
ℹ️  Reached last page.
Total posts collected: 229


### Results Preview 

In [10]:
# First 5 rows
df.head()

,title,selftext,created,ups,subreddit,url
0,Dapat bang sunugin ang damit kung nakita kang ...,"Pag nakita ka bang walang ulo, kailangang sunu...",2026-05-15 09:45:30 UTC,3,TrueScaryStories,https://reddit.com/r/TrueScaryStories/comments...
1,"Napatunayan kong hindi palaging kailangan ng ""...","Mga ka-PHGov, gusto ko lang i-share itong nagi...",2026-05-14 15:06:40 UTC,111,PHGov,https://reddit.com/r/PHGov/comments/1tcqg2j/na...
2,mga kamaganak na mahilig mangutang,"I have this uncle, laging pumupunta rito para ...",2026-05-12 21:03:41 UTC,1,RantAndVentPH,https://reddit.com/r/RantAndVentPH/comments/1t...
3,I'm tired,Hello. Just want to share. Please sana walang ...,2026-05-12 00:50:08 UTC,15,nanayconfessions,https://reddit.com/r/nanayconfessions/comments...
4,internship sa start-up moments,honestly sobrang pagod na ako mentally and emo...,2026-05-11 21:21:44 UTC,1,RantAndVentPH,https://reddit.com/r/RantAndVentPH/comments/1t...


In [11]:
# Upvote distribution
df["ups"].describe()

count    229.000000
mean      35.812227
std      104.314406
min        0.000000
25%        1.000000
50%        4.000000
75%       19.000000
max      874.000000
Name: ups, dtype: float64

In [12]:
# Top 10 posts by upvotes
df.sort_values("ups", ascending=False)[["title", "subreddit", "ups", "created"]].head(10)

,title,subreddit,ups,created
90,Mànay Saling,phhorrorstories,874,2026-03-17 02:53:56 UTC
155,Oh to be loved,MayNagChat,625,2026-01-24 11:08:51 UTC
19,"We can have more in life, because we can becom...",adultingphwins,620,2026-05-01 22:52:03 UTC
226,my furrbaby is positive of ehrlichia,dogsofrph,471,2025-12-03 21:30:41 UTC
52,GSM experience: nagbook ako nito para safe ako...,adultingph,431,2026-04-12 02:23:51 UTC
46,Egg drop soup pinalevel up nilagyan ko ng miswa 😋,PHFoodPorn,386,2026-04-14 09:17:13 UTC
59,Ginawa akong character reference ng agent ko n...,BPOinPH,342,2026-04-06 21:48:16 UTC
153,"I'm afraid, I feel like I don't deserve my boy...",phlgbt,317,2026-01-25 23:21:00 UTC
177,Gigil ako sa animal na ito,GigilAko,288,2026-01-08 23:09:01 UTC
203,I am starting to hate white Americans and I wa...,buhaydigital,265,2025-12-19 21:24:16 UTC


In [13]:
# Post count by subreddit
df["subreddit"].value_counts().head(10)

subreddit
RantAndVentPH       35
OffMyChestPH        19
adviceph            18
phhorrorstories     10
singleph             8
OALangBaAko          8
nanayconfessions     6
CasualPH             6
MayNagChat           5
MentalHealthPH       4
Name: count, dtype: int64

### Save to CSV

In [14]:
save_results(df, OUTPUT_FILE)

💾 Appended to /Users/angelodelapaz/Documents/GitHub/HealthPH-Plus/data/raw/reddit/reddit_results.csv  (+202 posts, skipped 27 duplicates)
📊 Total rows in file: 5439


In [15]:
df.sample(10)

,title,selftext,created,ups,subreddit,url
0,Dapat bang sunugin ang damit kung nakita kang ...,"Pag nakita ka bang walang ulo, kailangang sunu...",2026-05-15 09:45:30 UTC,3,TrueScaryStories,https://reddit.com/r/TrueScaryStories/comments...
204,bakit ang OA ng mga lalaki,"napansin ko lang, karamihan sa mga lalaki tala...",2025-12-18 22:31:49 UTC,0,RantAndVentPH,https://reddit.com/r/RantAndVentPH/comments/1p...
86,Am I a bad daughter if I want to live on my own?,Problem/Goal: Palaging nagagalit (or should I ...,2026-03-20 18:56:42 UTC,12,adviceph,https://reddit.com/r/adviceph/comments/1ryt422...
220,Ano mas magandang excuse para di makasama sa r...,a. Hassle magbyahe pauwi dahil di 24hrs ang by...,2025-12-07 20:25:31 UTC,1,TanongLang,https://reddit.com/r/TanongLang/comments/1pgh5...
12,Bakit sila ganito?,Maliit na tindahan lang kami. Akala ata nila k...,2026-05-06 19:52:39 UTC,0,pinoy,https://reddit.com/r/pinoy/comments/1t5als6/ba...
20,Ang bittersweet sa part na mas may concern pa ...,Gagabihin ako sa work ko. Masama pakiramdam ko...,2026-05-01 15:37:26 UTC,22,OffMyChestPH,https://reddit.com/r/OffMyChestPH/comments/1t0...
137,TLDR: Hubby spends so much time with his paren...,Hi mommies! Hope everyone is doing well. Pleas...,2026-02-06 22:48:26 UTC,2,nanayconfessions,https://reddit.com/r/nanayconfessions/comments...
221,"My(37M) then, partner (22M) asked me to stay a...","2 years na kami, LDR, pero from QC ako and Cav...",2025-12-07 09:29:10 UTC,0,relationship_advicePH,https://reddit.com/r/relationship_advicePH/com...
54,Re: Boards 101 (from a retakers perspective),Original post from a month ago - Boards 101 - ...,2026-04-10 21:28:39 UTC,38,ExpertMDph,https://reddit.com/r/ExpertMDph/comments/1shmk...
165,Open letter for Miss A,"I'm L, married with no kids. Unang beses ko na...",2026-01-16 14:17:27 UTC,0,PinoyUnsentLetters,https://reddit.com/r/PinoyUnsentLetters/commen...
